In [ ]:
!pip install sentence-transformers scikit-learn plotly pandas google-genai

In [ ]:
import getpass
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import plotly.express as px
from google import genai

# 1. Securely Input Gemini API Key
api_key = getpass.getpass("Paste your Google Gemini API Key: ")
client = genai.Client(api_key=api_key)

# 2. Load Customer Feedback Dataset (Simulating qualitative inputs)
feedback_data = [
    "App crashes every time I try to checkout on iOS.",
    "The payment gateway fails on mobile web with card errors.",
    "Checkout button is unresponsive on safari browser.",
    "Love the product design, colors look amazing!",
    "UI is super slick and pleasant to navigate.",
    "Great aesthetic improvements in the latest release.",
    "Customer support took 5 days to respond to my billing issue.",
    "Refunds are impossible to get through support tickets.",
    "Support team was completely unhelpful with my account issue.",
    "Exporting CSV reports takes almost 3 minutes now.",
    "Dashboard loads extremely slowly when fetching annual metrics.",
    "App performance has degraded significantly since update.",
    "Search bar yields no results for valid invoice numbers.",
    "Filter by date option is missing on the mobile app.",
    "Can't find my saved addresses in the new menu layout."
]

df = pd.DataFrame({"feedback": feedback_data})

# 3. Generate Local Sentence Embeddings (Vector Representations)
print("1/4: Generating vector embeddings for qualitative feedback...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(df['feedback'].tolist())

# 4. Perform Mathematical Clustering (K-Means)
num_clusters = 4
print(f"2/4: Grouping feedback into {num_clusters} semantic clusters...")
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
df['cluster_id'] = kmeans.fit_predict(embeddings)

# 5. Gemini API Auto-Labeling & Sentiment Scoring
print("3/4: Querying Gemini API to summarize, label, and score clusters...")
theme_map = {}

for cluster_num in range(num_clusters):
    cluster_items = df[df['cluster_id'] == cluster_num]['feedback'].tolist()
    formatted_items = "\n".join([f"- {item}" for item in cluster_items])

    prompt = f"""
You are a Principal Product Researcher analyzing a semantically clustered batch of raw customer feedback:

{formatted_items}

Analyze this cluster and return EXACTLY three lines of text:
Theme Name: <A concise 3 to 5 word title representing this cluster>
Priority Level: <Low / Medium / High / Critical>
Executive Summary: <1-sentence summary of the core user pain point or feedback>
"""

    response = client.models.generate_content(
        model='gemini-3-flash-preview',
        contents=prompt
    )

    result_text = response.text.strip()
    print(f"\n--- SEMANTIC CLUSTER {cluster_num} ANALYSIS ---")
    print(result_text)

    # Extract Theme Name for the Plotly Legend
    theme_line = [line for line in result_text.split('\n') if line.startswith('Theme Name:')]
    if theme_line:
        theme_map[cluster_num] = theme_line[0].replace('Theme Name:', '').strip()
    else:
        theme_map[cluster_num] = f"Cluster {cluster_num}"

# Map AI-Generated Theme Titles back to DataFrame
df['Theme'] = df['cluster_id'].map(theme_map)

# 6. Reduce Vector Dimensions to 2D for Visual Plotting (PCA)
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)
df['x'] = coords[:, 0]
df['y'] = coords[:, 1]

# 7. Render Interactive Plotly Visual Map (Colab Explicit Renderer)
print("\n4/4: Rendering Interactive Visual Cluster Map...")
fig = px.scatter(
    df,
    x='x',
    y='y',
    color='Theme',
    hover_data=['feedback'],
    title="Customer Feedback Semantic Clustering Map (Gemini API Auto-Labeled)",
    labels={'Theme': 'Thematic Cluster'},
    template="plotly_dark"
)

fig.update_traces(marker=dict(size=14, line=dict(width=1, color='White')))

# Force Google Colab to display the Plotly HTML object
fig.show(renderer="colab")

Paste your Google Gemini API Key: ··········
1/4: Generating vector embeddings for qualitative feedback...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2/4: Grouping feedback into 4 semantic clusters...
3/4: Querying Gemini API to summarize, label, and score clusters...

--- SEMANTIC CLUSTER 0 ANALYSIS ---
Theme Name: Enhanced UI and Visual Appeal
Priority Level: Low
Executive Summary: Users are highly satisfied with the recent aesthetic upgrades and the intuitive navigation of the user interface.

--- SEMANTIC CLUSTER 1 ANALYSIS ---
Theme Name: Critical Performance and Navigation Regressions
Priority Level: Critical
Executive Summary: Recent updates have introduced significant system latency and UI discoverability issues that are severely hindering core user workflows.

--- SEMANTIC CLUSTER 2 ANALYSIS ---
Theme Name: Critical Transaction and Billing Friction
Priority Level: Critical
Executive Summary: Users are facing systemic technical failures during checkout compounded by unresponsively slow support for billing and refund issues.

--- SEMANTIC CLUSTER 3 ANALYSIS ---
Theme Name: Core Functionality and Support Deficiencies
Priority 